# IndicVoices — Indian Multilingual Speech Dataset Analysis

**Dataset:** [ai4bharat/IndicVoices](https://huggingface.co/datasets/ai4bharat/IndicVoices)  
**Size:** 12,000+ hours, 22 Indian languages, 22,563 speakers, 400+ districts  
**License:** CC BY 4.0 (commercially usable)  
**Relevance to Tasknova:** Foundation for Hindi-English ASR, Indian accent coverage, Whisper fine-tuning  

This notebook loads a subset of the dataset, runs sanity checks, and explores language/speaker/duration distributions.

## 1. Setup & Installation

In [ ]:
!pip install -q datasets pandas matplotlib seaborn librosa soundfile

In [ ]:
import os
os.environ["HF_TOKEN"] = "YOUR_HF_TOKEN_HERE"

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")
pd.set_option('display.max_colwidth', 100)

In [ ]:
from datasets import load_dataset

def try_load_streaming(split="train"):
    """Try loading with token first, fall back to no token."""
    for kwargs in [{"token": os.environ["HF_TOKEN"]}, {}]:
        try:
            ds = load_dataset("ai4bharat/IndicVoices", streaming=True, split=split, **kwargs)
            _ = next(iter(ds))  # confirm it works
            return load_dataset("ai4bharat/IndicVoices", streaming=True, split=split, **kwargs)
        except Exception:
            continue
    return None

ds_stream = try_load_streaming()
if ds_stream:
    sample = next(iter(ds_stream))
    print("=== Column names ===")
    print(list(sample.keys()))
    print("\n=== Sample record (truncated) ===")
    for k, v in sample.items():
        if k == 'audio':
            if isinstance(v, dict):
                print(f"  {k}: sampling_rate={v.get('sampling_rate')}, array_len={len(v.get('array', []))}")
            else:
                print(f"  {k}: {type(v)}")
        else:
            print(f"  {k}: {str(v)[:150]}")
else:
    print("ERROR: Could not load dataset with or without token.")
    print("Visit https://huggingface.co/datasets/ai4bharat/IndicVoices to request access.")

In [ ]:
import pandas as pd
from pathlib import Path

SAMPLE_SIZE = 5_000
LOCAL_CACHE = Path("data/indicvoices_sample.parquet")

if LOCAL_CACHE.exists():
    df = pd.read_parquet(LOCAL_CACHE)
    print(f"Loaded {len(df):,} records from local cache: {LOCAL_CACHE}")
    print(f"Columns: {list(df.columns)}")
else:
    records = []
    for kwargs in [{"token": os.environ["HF_TOKEN"]}, {}]:
        try:
            ds_stream = load_dataset(
                "ai4bharat/IndicVoices", streaming=True, split="train", **kwargs
            )
            for i, item in enumerate(ds_stream):
                if i >= SAMPLE_SIZE:
                    break
                row = {}
                for k, v in item.items():
                    if k == 'audio' and isinstance(v, dict):
                        row['sampling_rate'] = v.get('sampling_rate')
                        arr = v.get('array', [])
                        sr = v.get('sampling_rate', 16000)
                        row['audio_duration_sec'] = len(arr) / sr if sr else 0
                        row['audio_samples'] = len(arr)
                    elif k != 'audio':
                        row[k] = v
                records.append(row)
            print(f"Loaded {len(records):,} records ({'with' if 'token' in kwargs else 'without'} token)")
            break  # success — stop trying
        except Exception as e:
            records = []
            if 'token' in kwargs:
                print(f"Token auth failed ({type(e).__name__}), retrying without token...")
            else:
                print(f"ERROR: {e}")

    if records:
        df = pd.DataFrame(records)
        LOCAL_CACHE.parent.mkdir(parents=True, exist_ok=True)
        df.to_parquet(LOCAL_CACHE, index=False)
        print(f"Saved to {LOCAL_CACHE}")
    else:
        print("Could not load dataset. Visit https://huggingface.co/datasets/ai4bharat/IndicVoices")
        df = pd.DataFrame()

if not df.empty:
    print(f"\nColumns: {list(df.columns)}")
    display(df.head(3))

In [ ]:
if df.empty:
    print("No data loaded — skipping checks.")
else:
    print("=" * 60)
    print("SANITY CHECK 1: Data types and shape")
    print("=" * 60)
    print(f"Shape: {df.shape}")
    print(f"\nData types:\n{df.dtypes}")
    print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

In [ ]:
if not df.empty:
    print("=" * 60)
    print("SANITY CHECK 2: Missing values")
    print("=" * 60)
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    missing_report = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
    print(missing_report)

In [ ]:
if not df.empty:
    print("=" * 60)
    print("SANITY CHECK 3: Audio integrity")
    print("=" * 60)

    if 'audio_duration_sec' in df.columns:
        zero_duration = (df['audio_duration_sec'] == 0).sum()
        very_short = (df['audio_duration_sec'] < 0.5).sum()
        very_long = (df['audio_duration_sec'] > 600).sum()
        print(f"Zero-duration audio: {zero_duration}")
        print(f"Very short (< 0.5s): {very_short}")
        print(f"Very long (> 10 min): {very_long}")
        print(f"\nDuration stats (seconds):")
        print(df['audio_duration_sec'].describe().round(2))

    if 'sampling_rate' in df.columns:
        print(f"\nSampling rates found: {df['sampling_rate'].unique()}")
        print(f"Sampling rate distribution:\n{df['sampling_rate'].value_counts()}")

In [ ]:
if not df.empty:
    print("=" * 60)
    print("SANITY CHECK 4: Transcript quality")
    print("=" * 60)

    transcript_col = None
    for candidate in ['transcript', 'text', 'sentence', 'transcription', 'utterance']:
        matches = [c for c in df.columns if candidate in c.lower()]
        if matches:
            transcript_col = matches[0]
            break

    if transcript_col:
        lengths = df[transcript_col].astype(str).str.len()
        empty = (lengths == 0).sum()
        na_count = df[transcript_col].isna().sum()
        print(f"Transcript column: '{transcript_col}'")
        print(f"  Empty transcripts: {empty}")
        print(f"  Null transcripts: {na_count}")
        print(f"  Median length: {lengths.median():.0f} chars")
        print(f"  Mean length: {lengths.mean():.0f} chars")
        print(f"  Max length: {lengths.max():,} chars")

        if 'audio_duration_sec' in df.columns:
            mask = df['audio_duration_sec'] > 0
            df.loc[mask, '_chars_per_sec'] = lengths[mask] / df.loc[mask, 'audio_duration_sec']
            print(f"\n  Chars/sec (speech rate proxy):")
            print(f"    Mean: {df['_chars_per_sec'].mean():.1f}")
            print(f"    Median: {df['_chars_per_sec'].median():.1f}")
            outliers = ((df['_chars_per_sec'] < 2) | (df['_chars_per_sec'] > 50)).sum()
            print(f"    Outliers (< 2 or > 50 chars/sec): {outliers}")
    else:
        print("No transcript column found.")

In [ ]:
if not df.empty:
    print("=" * 60)
    print("SANITY CHECK 5: Categorical field distributions")
    print("=" * 60)

    hashable_cols = [c for c in df.columns if df[c].apply(lambda x: not isinstance(x, list)).all()]
    for col in hashable_cols:
        if col.startswith('_') or col in ('audio_samples',):
            continue
        n_unique = df[col].nunique()
        if 2 <= n_unique <= 50:
            print(f"\n'{col}' — {n_unique} unique values:")
            print(df[col].value_counts().head(25))

# Detect key columns once — used by all chart cells below
lang_col = next((c for cand in ['language', 'lang', 'locale']
                 for c in df.columns if cand in c.lower()), None) if not df.empty else None
type_col = next((c for cand in ['type', 'speech_type', 'style', 'task', 'category']
                 for c in df.columns if cand in c.lower()), None) if not df.empty else None
gender_col = next((c for cand in ['gender', 'sex']
                   for c in df.columns if cand in c.lower()), None) if not df.empty else None
region_col = next((c for cand in ['state', 'district', 'region', 'location']
                   for c in df.columns if cand in c.lower()), None) if not df.empty else None

if not df.empty and lang_col:
    fig, ax = plt.subplots(figsize=(10, 8))
    counts = df[lang_col].value_counts()
    counts.plot(kind='barh', ax=ax, color='teal')
    ax.set_title(f'Language Distribution (n={len(df):,})')
    ax.set_xlabel('Count')
    for i, v in enumerate(counts.values):
        ax.text(v + len(df)*0.005, i, f"{v:,} ({v/len(df)*100:.1f}%)", va='center', fontsize=8)
    plt.tight_layout()
    plt.show()
elif not df.empty:
    print("No language column found.")

In [ ]:
if not df.empty and 'audio_duration_sec' in df.columns:
    durations = df['audio_duration_sec'][df['audio_duration_sec'] > 0]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].hist(durations[durations < 120], bins=60, color='darkorange', edgecolor='white')
    axes[0].set_title('Audio Duration Distribution (< 2 min)')
    axes[0].set_xlabel('Duration (seconds)')
    axes[0].set_ylabel('Count')
    axes[0].axvline(durations.median(), color='red', linestyle='--',
                    label=f"Median: {durations.median():.1f}s")
    axes[0].legend()

    if lang_col:
        hours_by_lang = df.groupby(lang_col)['audio_duration_sec'].sum() / 3600
        hours_by_lang.sort_values(ascending=True).plot(kind='barh', ax=axes[1], color='steelblue')
        axes[1].set_title('Total Hours per Language (in sample)')
        axes[1].set_xlabel('Hours')

    plt.tight_layout()
    plt.show()

    total_hrs = durations.sum() / 3600
    print(f"Total audio in sample: {total_hrs:.1f} hours")
    print(f"Mean duration: {durations.mean():.1f}s | Median: {durations.median():.1f}s")

In [ ]:
if not df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    if gender_col:
        df[gender_col].value_counts().plot(kind='pie', autopct='%1.1f%%', ax=axes[0],
                                           colors=['#4C72B0', '#DD8452', '#55A868'])
        axes[0].set_title('Gender Distribution')
        axes[0].set_ylabel('')
    else:
        axes[0].text(0.5, 0.5, 'No gender column', ha='center', va='center')
        axes[0].set_title('Gender Distribution')

    if region_col:
        df[region_col].value_counts().head(15).plot(kind='barh', ax=axes[1], color='steelblue')
        axes[1].set_title(f'Top 15 Regions ({region_col})')
        axes[1].set_xlabel('Count')
    else:
        axes[1].text(0.5, 0.5, 'No region column', ha='center', va='center')
        axes[1].set_title('Regional Distribution')

    plt.tight_layout()
    plt.show()

if not df.empty and lang_col and type_col:
    ct = pd.crosstab(df[lang_col], df[type_col])
    print("Language x Speech Type crosstab:")
    print(ct)

    fig, ax = plt.subplots(figsize=(12, 8))
    ct.plot(kind='barh', stacked=True, ax=ax, colormap='Set2')
    ax.set_title('Speech Type by Language')
    ax.set_xlabel('Count')
    plt.tight_layout()
    plt.show()

if not df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    if gender_col:
        df[gender_col].value_counts().plot(kind='pie', autopct='%1.1f%%', ax=axes[0],
                                           colors=['#4C72B0', '#DD8452', '#55A868'])
        axes[0].set_title('Gender Distribution')
        axes[0].set_ylabel('')
    else:
        axes[0].text(0.5, 0.5, 'No gender column', ha='center', va='center')
        axes[0].set_title('Gender Distribution')

    if region_col:
        df[region_col].value_counts().head(15).plot(kind='barh', ax=axes[1], color='steelblue')
        axes[1].set_title(f'Top 15 Regions ({region_col})')
        axes[1].set_xlabel('Count')
    else:
        axes[1].text(0.5, 0.5, 'No region column', ha='center', va='center')
        axes[1].set_title('Regional Distribution')

    plt.tight_layout()
    plt.show()

if not df.empty and lang_col and type_col:
    ct = pd.crosstab(df[lang_col], df[type_col])
    print("Language x Speech Type crosstab:")
    print(ct)

    fig, ax = plt.subplots(figsize=(12, 8))
    ct.plot(kind='barh', stacked=True, ax=ax, colormap='Set2')
    ax.set_title('Speech Type by Language')
    ax.set_xlabel('Count')
    plt.tight_layout()
    plt.show()